In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Read the dataset Q1_data.csv using read_csv()
csv_file_path = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(csv_file_path)

In [ ]:
# Task 2: Write your code here:
# Inspect the first few rows using head()
df.head() ## we can specify number of rows if we want 3 then pass three to .head(3)

In [ ]:
# Task 3: Write your code here:
# Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:
# Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: Write your code here:
# Drop the 'Delivery_Time' column from the data
target_column = "Delivery_Time"

X = df.drop(target_column, axis=1)
y = df[target_column]
X

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
  # Missing values
print("Missing values:")
print(df.isnull().sum())

df['Weather'].dtype


# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values,
    'column_type' : missing_percentage.dtype
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

  # Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
print(f"Before: {df.shape}")
df_clean = df.dropna(subset=['Delivery_Time'])
print(f"After dropping missing Delivery Time: {df_clean.shape}, before dropping it was {df.shape})")

  # Weather	Traffic_Level	Time_of_Day are object type I can choose to fill it with mode rather than will with Unknown
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

  # Courier_Experience_yrs     is a numerical column I think mean will be good to fill it # df['col'].fillna(df['col'].mean())
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())

df_clean.isna().sum()

In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates if any exist
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.shape
num_Col = ['Order_ID','Distance_km'	,'Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
cat_df = df_clean.drop(num_Col, axis=1)
cat_df.shape

In [ ]:

num_df = df_clean.drop(cat_df, axis=1)
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(cat_df), columns=onehot_encoder.get_feature_names_out(cat_df.columns))
X_encoded
num_df = df_clean.drop(cat_df, axis=1)
num_df.shape
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(cat_df), columns=onehot_encoder.get_feature_names_out(cat_df.columns))
X_encoded.isna().sum()
df_enc = pd.concat([X_encoded,num_df], axis = 1)
df_enc.isna().sum() # i got 106 null after doing this so i will remove them .dropna()
df_enc = df_enc.dropna()
df_enc.isna().sum()


In [ ]:
# # Task 4: Write your code here:
# # Encode categorical variables if needed (Bonus if used One Hot Encoding)

# num_Col = ['Order_ID','Distance_km'	,'Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
# cat_df = df_clean.drop(num_Col, axis=1)
# cat_df
# num_df = df_clean.drop(cat_df, axis=1)
# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# X_encoded = pd.DataFrame(onehot_encoder.fit_transform(cat_df), columns=onehot_encoder.get_feature_names_out(cat_df.columns))
# X_encoded

# df_enc = pd.concat([X_encoded,num_df], axis = 1)
# df_enc

In [ ]:
# # # Task 4: Write your code here:
# # # Encode categorical variables if needed (Bonus if used One Hot Encoding)

# # onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
# # data_onehot_encoded = onehot_encoder.fit_transform(df_clean) # Apply fit_transform to the copied
# # data_onehot_encoded,df_clean

# print("Applying One-Hot Encoding to feature DataFrame 'X'...")
# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# X_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_clean), columns=onehot_encoder.get_feature_names_out(df_clean.columns))
# X_encoded
# # ## first seperate X and y(target) to avoid data leakage
# # # target_column = "Delivery_Time"

# # # X = df_clean.drop(target_column, axis=1)
# # # y = df_clean[target_column]

# # # Encode categorical columns - converts text to integers

# # categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
# # for col in categorical_cols:
# #     onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False,dtype=str)
# #     # df_clean[col] = onehot_encoder.fit_transform(df_clean[col].astype(str))
# #     X_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_clean[col].view(1,-1)), columns=onehot_encoder.get_feature_names_out(df_clean[col].columns))

# # df_clean.head()





In [ ]:
# categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

# for col in categorical_cols:
#   onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#   df_clean[col] = onehot_encoder.fit_transform(df_clean[col].astype(str))

#   print(df_clean[col])

# ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.


In [ ]:
# num_Col = ['Order_ID','Distance_km'	,'Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
# cat_df = df_clean.drop(num_Col, axis=1)
# cat_df
# for col in cat_df:
#     onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#     # df_clean[col] = onehot_encoder.fit_transform(df_clean[col].astype(str))
#     cat_df_encoded = pd.DataFrame(onehot_encoder.fit_transform(col).view(1, -1))


# # ValueError: Expected 2D array, got scalar array instead:
# # array=Weather.
# # Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
# num_Col = ['Order_ID','Distance_km'	,'Preparation_Time_min','Courier_Experience_yrs','Delivery_Time']
# cat_df = df_clean.drop(num_Col, axis=1)
# cat_df
# # for col in cat_df:
# #     onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# #     # df_clean[col] = onehot_encoder.fit_transform(df_clean[col].astype(str))
# #     cat_df_encoded = pd.DataFrame(onehot_encoder.fit_transform(col))
# #     print("Applying One-Hot Encoding to feature DataFrame 'X'...")
# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# X_encoded = pd.DataFrame(onehot_encoder.fit_transform(cat_df), columns=onehot_encoder.get_feature_names_out(cat_df.columns))
# X_encoded

In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
# Scale features - fit on train, transform both
X = df_enc.drop(target_column, axis=1)
y = df_enc[target_column]

## why u want us to scale then split this will cause data leakage!
# I will do the split here sorry if my model will not perform as my coleages

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
import seaborn as sns
print("Target Distribution:")
print(y.value_counts(normalize=True))
sns.countplot(x=y)
plt.title("Target Distribution")
plt.show()


# i dont know i think everything is ok our target is numerical not categorical this make me strugle to decide

In [ ]:
# Task 1: Write your code here:
# Split the dataset into features (X) and target (y)

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# already split above

In [ ]:
# Rule of thumb: If class distributions are not equal, then our data is imbalanced. Use StratifiedKFold and focus on F1-score. And if data distribution is balanced, StratifiedKFold will act like regular KFold, so always use StratifiedKFold :)
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
# Train
mae = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=200)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate


    # 3. Save metrics for that model in this

  mae_fold = mean_absolute_error(y_test, y_pred)
  mae.append(mae_fold)
    # accuracy = accuracy_score(y_test, y_pred)
    # precision = precision_score(y_test, y_pred, zero_division=0)
    # recall = recall_score(y_test, y_pred, zero_division=0)
    # f1 = f1_score(y_test, y_pred, zero_division=0)

    # all_results[model_name]['accuracy'].append(accuracy)
    # all_results[model_name]['precision'].append(precision)
    # all_results[model_name]['recall'].append(recall)
    # all_results[model_name]['f1'].append(f1)

In [ ]:
import numpy as np
avg_loss = np.mean(mae, axis=0)


In [ ]:
print(avg_loss)

In [ ]:
# # Task 2,3,4,5: Write your code here:
# # Use the correct split: KFold OR StratifiedKFold
# # Train a RandomForest model
# # Evaluate using MAE (Mean Absolute Error) ONLY
# # Print the averaged score across all folds
# from sklearn.metrics import mean_absolute_error, mean_squared_error
# from sklearn.ensemble import RandomForestRegressor
# model = RandomForestRegressor(n_estimators=200)
# # Train
# model.fit(X_train, y_train)
# # Predict
# y_pred = model.predict(X_test)
# # Calculate metrics
# mae = mean_absolute_error(y_test, y_pred)
# print(mae) >avg_loss

In [ ]:
# Task 1: Write your code here:
# Plot feature importance from your trained model

# Gather importances from the models (from the last fold)
importances = {}

importances['RandomForestRegressor'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
#Plot predicted delivery time histogram

# Plot for Random Forest Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)],
         [min(y_test), max(y_test)],
         'r--', linewidth=2)

plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Decision Tree)")
plt.title("Random Forest Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:

